# Actuals from the VCR CSV - the P&L columns

Builds the actuals for the segment heat map out of the VCR extract you already
have, in Python. No new SQL.

Two stages, in the order the spec document lays them out:

1. **The monthly P&L** - the thirteen revenue lines, Miscellaneous Fees/Income,
   Total Gross Revenues, the provision, Total Net Revenues, the six servicing
   expense lines, the other expenses (cost of funds, booking, recovery, fraud,
   CPA, Amex incentive), Total Operating Expenses and Pretax Income.
2. **The cumulative measures** the heat map reads - Cumul. Pre-Tax Income,
   Cumul. Net Charge-Off Before Recoveries, Cumul. ROA Annualized and the
   Cumul. Unit Charge-Off Rate, each running across MOB inside one vintage
   cohort.

Every line names the CSV column it reads, in the spec cell below. Two things the
document asks for are **not in the extract** - `CPA` and `amex_share` - so those
two lines read zero until you set them in the settings cell.

## Settings

In [ ]:
# The VCR extract. Point this at your file.
CSV_PATH = "vcr_actuals.csv"
OUT_CSV  = "vcr_with_pnl.csv"

# One Time - Marketing Expense (CPA). The extract has no CPA column, so give one
# of these:
#   a column name    e.g. "MarketingCost"   - read straight from that column
#   a number         e.g. 185.0             - dollars per board, charged in the
#                                             booking month
#   None                                    - the line reads zero
CPA = None

# The amex_share in "2.03 * amex_share * booked accounts / 24". Not in the
# extract. None leaves the Amex Incentive line at zero.
AMEX_SHARE = None

# Denominator for the Cumul. Unit Charge-Off Rate:
#   "active"             active accounts (ActiveAccountIndicator) - moves with
#                        MOB, so the rate rises on both a bigger numerator and
#                        a shrinking book
#   "boards_net_unwind"  booked accounts less unwinds through MOB 4 - constant
#                        across MOB
UNIT_CO_DENOMINATOR = "active"

## The spec

Transcribed from the spec document. Each line names its CSV column; the two
gaps are marked.

In [ ]:
"""The P&L lines, transcribed from the spec document.

Nothing here is inferred: each line names the CSV column it reads. Where the
document gives a formula instead of a column it is marked DERIVED, and where it
asks for something the CSV does not carry it is marked MISSING and reads zero
until you point it at a source.
"""

# ---------------------------------------------------------------- revenues --
# sign +1 adds to gross revenue, -1 is a contra shown in brackets in the doc
REVENUE_LINES = [
    ("Enrollment Fees",        "FeeEnrollmentAmount",      +1),
    ("Credit Protection Fees", "CreditProtectionAmount",   +1),
    ("Annual Fees",            "FeeAnnualChargeAmount",    +1),
    ("Finance Charges",        "TotalFinanceCharges",      +1),
    ("Late Fees",              "FeeLateAccruedAmount",     +1),
    ("Overlimit Fees",         "FeeOverlimitAmount",       +1),
    ("NSF Fees",               "FeeMiscAccruedTtlAmt",     +1),
    ("Interchange Income",     "InterchangeRevenue",       +1),
    ("Direct Check Fees",      "FeeDirectCheckAmount",     +1),
    ("Cash Advance Fees",      "FeeCashAdvanceAmount",     +1),
    ("CLI Fees",               "FeeCrLineIncreaseMiscAmt", +1),
    ("(Rewards)",              "RewardsAmount",            -1),
    ("(Fee Reversals)",        "TotalFeeAdjustments",      -1),
]
# DERIVED: "Gross Revenue - sum(all other revenues except net revenue)", so the
# named lines plus this one always add back to GrossRevenue exactly.
MISC_REVENUE_LINE = "Miscellaneous Fees/Income"

GROSS_REVENUE   = "GrossRevenue"        # Total Gross Revenues
PROVISION       = "ProvisionExpense"    # (Provision for Loan Loss)
# Total Net Revenues = Gross Revenue + (Provision Expense). The brackets make it
# a deduction, so net = gross - provision when provision is carried positive.
# Set to -1 if your CSV already carries provision as a negative number.
PROVISION_SIGN = -1

# ---------------------------------------------------------------- expenses --
SERVICING_EXPENSE_LINES = [
    ("Customer Service",             "CustomerServiceExpense"),
    ("Collections (Pre-Call, 5-30)", "CollectionsExpensePreCalland530"),
    ("Collections (60+)",            "CollectionsExpense60Plus"),
    ("Systems",                      "SystemsExpense"),
    ("Credit Protection Expense",    "CreditProtectionExpense"),
    ("Other Overhead",               "OtherOverheadExpense"),
]

OTHER_EXPENSE_LINES = [
    ("Cost of Funds - Debt Expense", "CostofFundsExpense"),
    ("One Time - Booking Expense",   "OneTimeBookingExpense"),
    ("Recovery Expense",             "RecoveryExpense"),
]
# DERIVED: Fraud Loss Expense = FraudGrossAmount88 + FraudNetAmount52
FRAUD_LOSS_PARTS = ["FraudGrossAmount88", "FraudNetAmount52"]

# MISSING: the CSV has no CPA column. Point CPA_COLUMN at one if it gains
# a marketing cost, or leave it None and the line reads zero.
CPA_COLUMN = None

# MISSING: "Amex Incentive Expense - if date >= feb 2024 and mob <= 24, then
# 2.03 * amex_share * booked accounts / 24, and negate it". amex_share is not in
# the CSV. Fill in AMEX_SHARE to switch the line on.
AMEX_INCENTIVE = dict(
    rate=2.03,
    share=None,                  # <- the amex_share the document means
    from_vintage="2024-02",      # date >= feb 2024
    max_mob=24,                  # and mob <= 24
    months=24,                   # the /24
)

# ------------------------------------------------------------------- rates --
# Cumulative net charge-off before recoveries
#     = ChargeOffPrincipalAmount / Avg Total Gross Receivables - End of Period
# Cumulative ROA - Annualized
#     = (Cumul. Pre-Tax Income / Avg Gross Receivables - EOP) / MOB * 12
# Cumulative Unit Charge-Off Rate
#     = ChargeOffIndicator / active accounts
# The other reading of the document's line is booked accounts less unwinds
# through MOB 4; keep it available with unit_co_denominator="boards_net_unwind".
UNIT_CO_DENOMINATOR = "active"
UNWIND_THROUGH_MOB = 4
RECEIVABLE_COLUMN  = "EndingReceivable"
BOARDS_COLUMN      = "NewAccounts"
UNWIND_COLUMN      = "UnwindCount"
CHARGE_OFF_UNITS   = "ChargeOffIndicator"
CHARGE_OFF_PRIN    = "ChargeOffPrincipalAmount"
ACTIVE_COLUMN      = "ActiveAccountIndicator"

# Expose the settings above to the builder below as `spec`, the way
# build_pnl.py imports them. Edit anything above and re-run this cell.
import types
spec = types.ModuleType("pnl_spec")
spec.__dict__.update({k: v for k, v in list(globals().items()) if k.isupper()})

## The arithmetic

`Miscellaneous Fees/Income` is Gross Revenue less the named revenue lines, so
the revenue block adds back to `GrossRevenue` exactly - the notebook asserts
that below. `Pretax Income` is Total Net Revenues less Total Operating
Expenses.

In [ ]:
import pandas as pd

# ----------------------------------------------------------- line labels ----
L_MISC  = spec.MISC_REVENUE_LINE
L_TGR   = "Total Gross Revenues"
L_PROV  = "(Provision for Loan Loss)"
L_TNR   = "Total Net Revenues"
L_TSE   = "Total Servicing Expenses"
L_CPA   = "One Time - Marketing Expense (CPA)"
L_FRAUD = "Fraud Loss Expense"
L_AMEX  = "Amex Incentive Expense"
L_TOE   = "Total Operating Expenses"
L_PTI   = "Pretax Income"

L_CUM_PTI   = "Cumul. Pre-Tax Income"
L_AVG_REC   = "Avg Total Gross Receivables - End of Period"
L_CUM_NCO   = "Cumul. Net Charge-Off Before Recoveries"
L_CUM_ROA   = "Cumul. ROA - Annualized"
L_CUM_UNIT  = "Cumul. Unit Charge-Off Rate"
L_BOARDS    = "Booked Accounts"
L_UNIT_DEN  = "Unit C/O Denominator"

# Dimensions that identify a cohort. A cumulative measure runs across MOB
# within one of these, so they must not include MonthsOnBooks.
GRAIN_KEYS = [
    "vintage", "VintageDate", "Branding", "Acquisitions", "Association",
    "source", "Subchannel", "AnnualFeeGroup", "Fee", "OriginalCreditLine",
    "VantageRange",
]
MOB_KEY = "MonthsOnBooks"


class SpecGap(Exception):
    """A column the spec document asks for is not in the CSV."""


def _col(df, name, *, missing):
    """The column as a float series, or zeros when it is absent."""
    if name in df.columns:
        return pd.to_numeric(df[name], errors="coerce").fillna(0.0)
    missing.append(name)
    return pd.Series(0.0, index=df.index)


def grain_keys(df):
    """The cohort keys actually present in this CSV, in spec order."""
    return [k for k in GRAIN_KEYS if k in df.columns]


# ------------------------------------------------------------- stage one ----
def add_pnl_lines(df, *, cpa=None, amex_share=None, report=None):
    """Add the monthly P&L lines. Returns a new frame.

    cpa         a column name, a per-board dollar amount, or None. The spec's
                "One Time - Marketing Expense (CPA)" reads a CPA column the CSV
                does not have; pass one of these to fill the line in.
    amex_share  the amex_share the Amex Incentive line multiplies by. None
                leaves that line at zero.
    report      a list that collects notes about gaps and checks.
    """
    out = df.copy()
    notes = report if report is not None else []
    missing = []

    # -- revenues ------------------------------------------------------------
    named_total = pd.Series(0.0, index=out.index)
    for label, column, sign in spec.REVENUE_LINES:
        line = _col(out, column, missing=missing)
        out[label] = sign * line
        named_total = named_total + out[label]

    gross = _col(out, spec.GROSS_REVENUE, missing=missing)
    # Gross Revenue less the named revenue lines, so the block always adds
    # back to Gross Revenue exactly.
    out[L_MISC] = gross - named_total
    out[L_TGR] = named_total + out[L_MISC]

    # -- provision and net revenue ------------------------------------------
    provision = _col(out, spec.PROVISION, missing=missing)
    out[L_PROV] = spec.PROVISION_SIGN * provision
    out[L_TNR] = out[L_TGR] + out[L_PROV]

    # -- servicing expenses --------------------------------------------------
    servicing = pd.Series(0.0, index=out.index)
    for label, column in spec.SERVICING_EXPENSE_LINES:
        out[label] = _col(out, column, missing=missing)
        servicing = servicing + out[label]
    out[L_TSE] = servicing

    # -- other expenses ------------------------------------------------------
    other = pd.Series(0.0, index=out.index)
    for label, column in spec.OTHER_EXPENSE_LINES:
        out[label] = _col(out, column, missing=missing)
        other = other + out[label]

    # Fraud Loss Expense = FraudGrossAmount88 + FraudNetAmount52
    fraud = pd.Series(0.0, index=out.index)
    for column in spec.FRAUD_LOSS_PARTS:
        fraud = fraud + _col(out, column, missing=missing)
    out[L_FRAUD] = fraud
    other = other + fraud

    # One Time - Marketing Expense (CPA)
    out[L_CPA], cpa_note = _cpa_line(out, cpa, missing)
    other = other + out[L_CPA]
    if cpa_note:
        notes.append(cpa_note)

    # Amex Incentive Expense
    out[L_AMEX], amex_note = _amex_line(out, amex_share, missing)
    other = other + out[L_AMEX]
    if amex_note:
        notes.append(amex_note)

    out[L_TOE] = out[L_TSE] + other
    out[L_PTI] = out[L_TNR] - out[L_TOE]

    if missing:
        notes.append("columns not in the CSV, read as zero: "
                     + ", ".join(sorted(set(missing))))
    return out


def _cpa_line(out, cpa, missing):
    zero = pd.Series(0.0, index=out.index)
    if cpa is None:
        column = spec.CPA_COLUMN
        if column is None:
            return zero, ("CPA: the spec's One Time - Marketing Expense (CPA) "
                          "has no column in the CSV, so the line is zero. Pass "
                          "cpa=<column name> or cpa=<dollars per board>.")
        return _col(out, column, missing=missing), None
    if isinstance(cpa, str):
        if cpa not in out.columns:
            raise SpecGap(f"cpa column {cpa!r} is not in the CSV")
        return pd.to_numeric(out[cpa], errors="coerce").fillna(0.0), None
    # a flat dollar amount per booked account, charged in the booking month
    boards = _col(out, spec.BOARDS_COLUMN, missing=missing)
    return float(cpa) * boards, f"CPA: charged at {float(cpa):,.2f} per board."


def _amex_line(out, share, missing):
    """2.03 * amex_share * booked accounts / 24, negated, for vintages from
    Feb 2024 and MOB <= 24."""
    cfg = spec.AMEX_INCENTIVE
    zero = pd.Series(0.0, index=out.index)
    if share is None:
        share = cfg.get("share")
    if share is None:
        return zero, ("Amex Incentive: amex_share is not in the CSV, so the "
                      "line is zero. Pass amex_share=<fraction>.")
    keys = grain_keys(out)
    boards = _col(out, spec.BOARDS_COLUMN, missing=missing)
    cohort_boards = boards.groupby([out[k] for k in keys]).transform("sum") \
        if keys else boards
    mob = _col(out, MOB_KEY, missing=missing)
    vdate = _vintage_month(out)
    live = (vdate >= pd.Period(cfg["from_vintage"], freq="M")) \
        & (mob <= cfg["max_mob"]) & (mob >= 1)
    amount = -cfg["rate"] * float(share) * cohort_boards / cfg["months"]
    return amount.where(live, 0.0), (
        f"Amex Incentive: {cfg['rate']} x {share} x boards / {cfg['months']}, "
        f"negated, for vintages from {cfg['from_vintage']} and MOB <= "
        f"{cfg['max_mob']}.")


def _vintage_month(df):
    for name in ("VintageDate", "vintage"):
        if name in df.columns:
            parsed = pd.to_datetime(df[name], errors="coerce")
            if parsed.notna().any():
                return parsed.dt.to_period("M")
    return pd.Series(pd.NaT, index=df.index).dt.to_period("M")


# ------------------------------------------------------------- stage two ----
def add_cumulative(df, *, unit_co_denominator=None, report=None):
    """Add the cumulative measures, running across MOB within each cohort."""
    notes = report if report is not None else []
    missing = []
    if unit_co_denominator is None:
        unit_co_denominator = spec.UNIT_CO_DENOMINATOR
    keys = grain_keys(df)
    if not keys:
        raise SpecGap("no cohort columns found; expected some of "
                      + ", ".join(GRAIN_KEYS))
    if MOB_KEY not in df.columns:
        raise SpecGap(f"{MOB_KEY} is not in the CSV")

    out = df.sort_values(keys + [MOB_KEY]).copy()
    g = out.groupby(keys, sort=False, dropna=False)
    mob = _col(out, MOB_KEY, missing=missing)

    # cumulative pretax income
    out[L_CUM_PTI] = g[L_PTI].cumsum()

    # average end-of-period gross receivables, to date
    rec = _col(out, spec.RECEIVABLE_COLUMN, missing=missing)
    out["_rec"] = rec
    out[L_AVG_REC] = out.groupby(keys, sort=False, dropna=False)["_rec"] \
        .expanding().mean().reset_index(level=list(range(len(keys))), drop=True)
    del out["_rec"]

    # cumulative net charge-off before recoveries
    out["_co_prin"] = _col(out, spec.CHARGE_OFF_PRIN, missing=missing)
    cum_co_prin = out.groupby(keys, sort=False, dropna=False)["_co_prin"].cumsum()
    del out["_co_prin"]
    out[L_CUM_NCO] = _div(cum_co_prin, out[L_AVG_REC])

    # cumulative ROA, annualised
    out[L_CUM_ROA] = _div(out[L_CUM_PTI], out[L_AVG_REC]) / mob.where(mob > 0) * 12

    # cumulative unit charge-off rate
    out["_co_unit"] = _col(out, spec.CHARGE_OFF_UNITS, missing=missing)
    cum_co_unit = out.groupby(keys, sort=False, dropna=False)["_co_unit"].cumsum()
    del out["_co_unit"]
    boards, denom = _unit_denominator(out, keys, unit_co_denominator, missing, notes)
    out[L_BOARDS] = boards
    out[L_UNIT_DEN] = denom
    out[L_CUM_UNIT] = _div(cum_co_unit, denom)

    if missing:
        notes.append("columns not in the CSV, read as zero: "
                     + ", ".join(sorted(set(missing))))
    return out.reset_index(drop=True)


def _unit_denominator(out, keys, mode, missing, notes):
    """Booked accounts less unwinds through MOB 4, or active accounts."""
    boards_m = _col(out, spec.BOARDS_COLUMN, missing=missing)
    boards = boards_m.groupby([out[k] for k in keys]).transform("sum")
    if mode == "active":
        active = _col(out, spec.ACTIVE_COLUMN, missing=missing)
        notes.append(f"Unit C/O rate over active accounts "
                     f"({spec.ACTIVE_COLUMN}), which moves with MOB.")
        if (active == 0).all():
            notes.append(f"WARNING - {spec.ACTIVE_COLUMN} is zero on every "
                         f"row, so the unit C/O rate is blank.")
        return boards, active
    if mode != "boards_net_unwind":
        raise SpecGap(f"unknown unit_co_denominator {mode!r}")
    mob = _col(out, MOB_KEY, missing=missing)
    unwind = _col(out, spec.UNWIND_COLUMN, missing=missing)
    early = unwind.where(mob <= spec.UNWIND_THROUGH_MOB, 0.0)
    unwinds = early.groupby([out[k] for k in keys]).transform("sum")
    notes.append(f"Unit C/O rate over booked accounts less unwinds through MOB "
                 f"{spec.UNWIND_THROUGH_MOB}.")
    return boards, boards - unwinds


def _div(num, den):
    den = den.where(den.notna() & (den != 0))
    return num / den


# ------------------------------------------------------------------ check ----
def reconcile(df, *, tolerance=0.01):
    """The revenue block must add back to GrossRevenue, and pretax income must
    equal net revenues less operating expenses. Returns the worst breaks."""
    checks = {}
    if spec.GROSS_REVENUE in df.columns:
        checks["Total Gross Revenues vs GrossRevenue"] = \
            (df[L_TGR] - pd.to_numeric(df[spec.GROSS_REVENUE],
                                       errors="coerce").fillna(0.0)).abs().max()
    checks["Pretax Income vs net revenues - operating expenses"] = \
        (df[L_PTI] - (df[L_TNR] - df[L_TOE])).abs().max()
    bad = {k: v for k, v in checks.items() if pd.notna(v) and v > tolerance}
    return checks, bad


PNL_COLUMNS = (
    [label for label, _c, _s in spec.REVENUE_LINES]
    + [L_MISC, L_TGR, L_PROV, L_TNR]
    + [label for label, _c in spec.SERVICING_EXPENSE_LINES]
    + [L_TSE]
    + [label for label, _c in spec.OTHER_EXPENSE_LINES]
    + [L_FRAUD, L_CPA, L_AMEX, L_TOE, L_PTI]
)
CUMULATIVE_COLUMNS = [L_CUM_PTI, L_AVG_REC, L_CUM_NCO, L_CUM_ROA,
                      L_BOARDS, L_UNIT_DEN, L_CUM_UNIT]


def build(csv_path, *, cpa=None, amex_share=None,
          unit_co_denominator=None, report=None):
    """Read the VCR CSV and add every column above."""
    notes = report if report is not None else []
    df = pd.read_csv(csv_path, low_memory=False)
    notes.append(f"{len(df):,} rows, {len(df.columns)} columns read from {csv_path}")
    df = add_pnl_lines(df, cpa=cpa, amex_share=amex_share, report=notes)
    df = add_cumulative(df, unit_co_denominator=unit_co_denominator, report=notes)
    _checks, bad = reconcile(df)
    for name, worst in bad.items():
        notes.append(f"CHECK FAILED - {name}: worst break {worst:,.2f}")
    return df

## Build

In [ ]:
log = []
df = build(CSV_PATH, cpa=CPA, amex_share=AMEX_SHARE,
           unit_co_denominator=UNIT_CO_DENOMINATOR, report=log)

checks, bad = reconcile(df)
print("reconciliation")
for name, worst in checks.items():
    flag = "FAILED" if name in bad else "ok"
    print(f"  [{flag}] {name}: worst break {worst:,.4f}")

print("\nnotes")
for line in log:
    print("  -", line)

print(f"\ncohort keys: {', '.join(grain_keys(df))}")

## The P&L for one cohort

Pick a vintage and a MOB and read the block down, the way the spec document
lays it out.

In [ ]:
import pandas as pd

PICK_MOB = 12

keys = grain_keys(df)
first = df[df[MOB_KEY] == PICK_MOB]
if first.empty:
    raise SystemExit(f"no rows at MOB {PICK_MOB}")
row = first.iloc[0]

print(" / ".join(f"{k}={row[k]}" for k in keys) + f"  MOB {PICK_MOB}\n")
SUBTOTALS = (L_TGR, L_TNR, L_TSE, L_TOE, L_PTI)
for label in PNL_COLUMNS:
    if label in SUBTOTALS:
        print(" " * 38 + "-" * 16)
    print(f"  {label:<36} {row[label]:>16,.2f}")

print("\ncumulative measures")
for label in CUMULATIVE_COLUMNS:
    value = row[label]
    pct = label in (L_CUM_NCO, L_CUM_ROA, L_CUM_UNIT)
    print(f"  {label:<44} {value:>12.2%}" if pct
          else f"  {label:<44} {value:>12,.2f}")

## The cumulative curve

The four measures by MOB, for the cohorts with the most boards - this is the
shape the heat map's Yr 1 / Yr 2 / Yr 3 actuals are read off, at MOB 12, 24
and 36.

In [ ]:
TOP_N = 6

keys = grain_keys(df)
size = (df.groupby(keys, dropna=False)[L_BOARDS].max()
          .sort_values(ascending=False).head(TOP_N))

for cohort in size.index:
    mask = pd.Series(True, index=df.index)
    for k, v in zip(keys, cohort if isinstance(cohort, tuple) else (cohort,)):
        mask &= df[k] == v
    sl = df[mask].sort_values(MOB_KEY)
    label = " / ".join(str(v) for v in
                       (cohort if isinstance(cohort, tuple) else (cohort,)))
    print(label)
    for mob in (12, 24, 36):
        r = sl[sl[MOB_KEY] == mob]
        if r.empty:
            print(f"    MOB {mob:>2}  -")
            continue
        r = r.iloc[0]
        print(f"    MOB {mob:>2}  unit C/O {r[L_CUM_UNIT]:>7.2%}"
              f"  (over {r[L_UNIT_DEN]:>8,.0f} of {r[L_BOARDS]:>8,.0f} boards)"
              f"   $ C/O {r[L_CUM_NCO]:>7.2%}"
              f"   ROA {r[L_CUM_ROA]:>8.2%}"
              f"   cum PTI {r[L_CUM_PTI]:>14,.0f}")
    print()

## Write it out

The enriched row-level file. The next step rolls this up to the heat map's
grain (`Branding` / `OriginalCreditLine` at `All`) and maps the names onto the
workbook's tabs.

In [ ]:
df.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}: {len(df):,} rows, {len(df.columns)} columns")
print(f"  {len(PNL_COLUMNS)} P&L columns + {len(CUMULATIVE_COLUMNS)} cumulative")